# Neural Network models for automatic segmentation of ventilation regimes during general anesthesia recovery phase

This notebook trains, evaluates, and compares three architectures on the 128 Hz anesthesia waveform dataset (`CO2`, `AWP`, `Cdyn`):
1. **1D U-Net** (dense sample-level segmentation baseline on 60s sliding windows)
2. **Breath-Tokenized BiGRU** (sequence-to-sequence recurrent model over breath-level physiological tokens)
3. **Breath-Tokenized Temporal Transformer** (self-attention encoder over breath-level physiological tokens)

In [ ]:
import os
import json
import time
import math
import numpy as np
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

## Setup config for Colab

In [ ]:
# Optional: Mount Google Drive so weights & dataset persist across Colab sessions
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/anesthesia_project"
else:
    BASE_DIR = "/content"

DATASET_DIR = os.path.join(BASE_DIR, "anesthesia_synth_dataset")
SAVE_DIR = os.path.join(BASE_DIR, "ventilation_models")
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {DEVICE}")
print(f"Saving models and evaluation outputs to: {SAVE_DIR}")

## Dataset loader & Breath Tokenizer
_Why division into 60 s windows for 1D U-Net?_
* Captures multi-breath cadence (8–15 breaths), apnea pauses (15–30s), and 8s compliance steps.
* Divisible cleanly by 256 ($7680 / 256 = 30$) across 4 pooling stages.

_Why Breath Tokenization for BiGRU and Transformer?_
* Compresses 64,000 raw 128 Hz samples into ~150 breath/pause tokens with 18 physiological morphology and variability features, letting the models attend across the entire recording in a single pass.

In [ ]:
class AnesthesiaWindowDataset(Dataset):
    """
    Slices full 500s recordings (3, 64000) into fixed-length windows (e.g., 60s = 7680 samples)
    and normalizes channels using strictly training-set mean & std.
    """
    def __init__(self, X, y, mean, std, window_samples=7680, stride_samples=3840):
        self.X = X
        self.y = y
        self.mean = mean[:, None]  # (3, 1)
        self.std = std[:, None]    # (3, 1)
        self.window_samples = window_samples

        self.indices = []
        n_cases, _, total_len = X.shape
        for c in range(n_cases):
            for start in range(0, total_len - window_samples + 1, stride_samples):
                self.indices.append((c, start))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        case_idx, start = self.indices[idx]
        end = start + self.window_samples
        x_win = (self.X[case_idx, :, start:end] - self.mean) / (self.std + 1e-6)
        y_win = self.y[case_idx, start:end]
        return torch.from_numpy(x_win).float(), torch.from_numpy(y_win).long()


FEATURE_NAMES = [
    "co2_max", "co2_min", "co2_mean", "co2_std",
    "co2_plateau_width_sec", "co2_cleft_depth",
    "awp_max", "awp_min", "awp_baseline", "awp_neg_dip", "awp_pos_area",
    "cdyn_mean", "cdyn_delta",
    "token_dur_sec", "inst_rr_bpm", "awp_co2_corr",
    "rolling_cv_rr", "rolling_cv_co2"
]
NUM_FEATURES = len(FEATURE_NAMES)


def segment_into_breath_boundaries(co2, awp, fs=128):
    """
    Finds breath boundaries (start_idx, end_idx) across a recording of length L.
    Uses CO2 troughs when CO2 is active, falls back to AWP peaks when CO2 is flat,
    and subdivides flatlines (>3.0s) into 2.0s tokens so Apnea/Artifacts stay dense.
    """
    n_samples = len(co2)
    co2_smooth = gaussian_filter1d(co2, sigma=int(0.15 * fs))
    awp_smooth = gaussian_filter1d(awp, sigma=int(0.12 * fs))

    co2_peaks, _ = find_peaks(co2_smooth, height=8.0, distance=int(1.2 * fs), prominence=5.0)
    boundaries = [0]

    if len(co2_peaks) >= 2:
        for i in range(len(co2_peaks) - 1):
            p1, p2 = co2_peaks[i], co2_peaks[i + 1]
            trough = p1 + int(np.argmin(co2_smooth[p1:p2]))
            boundaries.append(trough)
    else:
        awp_peaks, _ = find_peaks(np.abs(awp_smooth), height=3.0, distance=int(1.2 * fs))
        for i in range(len(awp_peaks) - 1):
            mid = (awp_peaks[i] + awp_peaks[i + 1]) // 2
            boundaries.append(mid)

    boundaries.append(n_samples)
    boundaries = sorted(list(set(boundaries)))

    max_token_len = int(3.0 * fs)
    sub_step = int(2.0 * fs)
    final_segments = []

    for s, e in zip(boundaries[:-1], boundaries[1:]):
        if e - s <= 0:
            continue
        if (e - s) > max_token_len:
            curr = s
            while curr < e:
                nxt = min(e, curr + sub_step)
                if e - nxt < int(0.5 * fs):
                    nxt = e
                final_segments.append((curr, nxt))
                curr = nxt
        else:
            final_segments.append((s, e))

    return final_segments


def extract_tokens_for_case(signals_3xL, labels_L=None, fs=128):
    """Converts raw waveforms (3, L) into (N_tokens, NUM_FEATURES) and token labels."""
    co2, awp, cdyn = signals_3xL
    segments = segment_into_breath_boundaries(co2, awp, fs=fs)
    n_tokens = len(segments)

    feats = np.zeros((n_tokens, NUM_FEATURES), dtype=np.float32)
    token_labels = np.zeros(n_tokens, dtype=np.int64) if labels_L is not None else None
    prev_cdyn = cdyn[0]

    for i, (s, e) in enumerate(segments):
        c_seg = co2[s:e]
        a_seg = awp[s:e]
        d_seg = cdyn[s:e]
        dur_s = max(1e-3, (e - s) / fs)

        c_max = float(np.max(c_seg))
        c_min = float(np.min(c_seg))
        c_mean = float(np.mean(c_seg))
        c_std = float(np.std(c_seg))

        if c_max > 8.0:
            high_mask = c_seg > (0.75 * c_max)
            plateau_w = float(np.sum(high_mask)) / fs
            active_idx = np.where(c_seg > 0.4 * c_max)[0]
            if len(active_idx) > int(0.5 * fs):
                mid_slice = c_seg[active_idx[0] + int(0.15 * fs) : active_idx[-1] - int(0.15 * fs)]
                cleft_depth = float(c_max - np.min(mid_slice)) if len(mid_slice) > 0 else 0.0
            else:
                cleft_depth = 0.0
        else:
            plateau_w = 0.0
            cleft_depth = 0.0

        a_max = float(np.max(a_seg))
        a_min = float(np.min(a_seg))
        a_base = float(np.percentile(a_seg, 20))
        a_neg_dip = float(a_min - a_base)
        a_pos_area = float(np.sum(np.maximum(0.0, a_seg - a_base))) / fs

        d_mean = float(np.mean(d_seg))
        d_delta = float(d_mean - prev_cdyn)
        prev_cdyn = d_mean
        inst_rr = 60.0 / dur_s if c_max > 8.0 or (a_max - a_min) > 3.0 else 0.0

        if np.std(a_seg) > 0.3 and np.std(c_seg) > 0.5:
            corr = float(np.corrcoef(a_seg, c_seg)[0, 1])
            if np.isnan(corr):
                corr = 0.0
        else:
            corr = 0.0

        feats[i, :16] = [
            c_max, c_min, c_mean, c_std, plateau_w, cleft_depth,
            a_max, a_min, a_base, a_neg_dip, a_pos_area,
            d_mean, d_delta, dur_s, inst_rr, corr
        ]

        if labels_L is not None:
            vals, counts = np.unique(labels_L[s:e], return_counts=True)
            token_labels[i] = vals[np.argmax(counts)]

    for i in range(n_tokens):
        w_start = max(0, i - 2)
        w_end = min(n_tokens, i + 3)
        rr_win = feats[w_start:w_end, 14]
        co2_win = feats[w_start:w_end, 0]
        feats[i, 16] = float(np.std(rr_win) / (np.mean(rr_win) + 1.0))
        feats[i, 17] = float(np.std(co2_win) / (np.mean(co2_win) + 1.0))

    return feats, token_labels, segments


def tokenize_split(X, y, fs=128):
    all_feats, all_labels, all_segments = [], [], []
    for i in range(len(X)):
        f, l, segs = extract_tokens_for_case(X[i], y[i], fs=fs)
        all_feats.append(f)
        all_labels.append(l)
        all_segments.append(segs)
    return all_feats, all_labels, all_segments


class BreathSequenceDataset(Dataset):
    def __init__(self, feats_list, labels_list, feat_mean, feat_std):
        self.feats = [(f - feat_mean) / (feat_std + 1e-6) for f in feats_list]
        self.labels = labels_list

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.feats[idx]).float(),
            torch.from_numpy(self.labels[idx]).long(),
        )


def collate_breath_sequences(batch):
    lengths = [item[0].size(0) for item in batch]
    max_len = max(lengths)
    feat_dim = batch[0][0].size(1)

    padded_x = torch.zeros((len(batch), max_len, feat_dim), dtype=torch.float32)
    padded_y = torch.full((len(batch), max_len), fill_value=-100, dtype=torch.long)
    pad_mask = torch.ones((len(batch), max_len), dtype=torch.bool)

    for i, (x, y) in enumerate(batch):
        L = x.size(0)
        padded_x[i, :L] = x
        padded_y[i, :L] = y
        pad_mask[i, :L] = False

    return padded_x, padded_y, pad_mask

## UNET standard model as reference

In [ ]:
class DoubleConv1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size=kernel_size, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class UNet1D(nn.Module):
    """
    4-level 1D U-Net for dense temporal segmentation of physiological waveforms.
    Input:  (B, 3, L) -> [CO2, AWP, Cdyn]
    Output: (B, num_classes, L)
    """
    def __init__(self, in_channels=3, num_classes=5, base_filters=32, dropout=0.1):
        super().__init__()
        f = base_filters

        # Encoder
        self.enc1 = DoubleConv1D(in_channels, f, kernel_size=7, dropout=dropout)
        self.pool1 = nn.MaxPool1d(4)  # /4

        self.enc2 = DoubleConv1D(f, f * 2, kernel_size=7, dropout=dropout)
        self.pool2 = nn.MaxPool1d(4)  # /16

        self.enc3 = DoubleConv1D(f * 2, f * 4, kernel_size=5, dropout=dropout)
        self.pool3 = nn.MaxPool1d(4)  # /64

        self.enc4 = DoubleConv1D(f * 4, f * 8, kernel_size=5, dropout=dropout)
        self.pool4 = nn.MaxPool1d(4)  # /256

        # Bottleneck (1 sample here = 2 seconds at 128 Hz)
        self.bottleneck = DoubleConv1D(f * 8, f * 16, kernel_size=5, dropout=dropout)

        # Decoder
        self.up4 = nn.ConvTranspose1d(f * 16, f * 8, kernel_size=4, stride=4)
        self.dec4 = DoubleConv1D(f * 16, f * 8, kernel_size=5, dropout=dropout)

        self.up3 = nn.ConvTranspose1d(f * 8, f * 4, kernel_size=4, stride=4)
        self.dec3 = DoubleConv1D(f * 8, f * 4, kernel_size=5, dropout=dropout)

        self.up2 = nn.ConvTranspose1d(f * 4, f * 2, kernel_size=4, stride=4)
        self.dec2 = DoubleConv1D(f * 4, f * 2, kernel_size=7, dropout=dropout)

        self.up1 = nn.ConvTranspose1d(f * 2, f, kernel_size=4, stride=4)
        self.dec1 = DoubleConv1D(f * 2, f, kernel_size=7, dropout=dropout)

        self.head = nn.Conv1d(f, num_classes, kernel_size=1)

    def _pad_and_cat(self, up_x, skip_x):
        diff = skip_x.size(-1) - up_x.size(-1)
        if diff > 0:
            up_x = F.pad(up_x, (0, diff))
        elif diff < 0:
            up_x = up_x[..., :skip_x.size(-1)]
        return torch.cat([skip_x, up_x], dim=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.dec4(self._pad_and_cat(self.up4(b), e4))
        d3 = self.dec3(self._pad_and_cat(self.up3(d4), e3))
        d2 = self.dec2(self._pad_and_cat(self.up2(d3), e2))
        d1 = self.dec1(self._pad_and_cat(self.up1(d2), e1))

        return self.head(d1)

## Breath-Tokenized BiGRU and Temporal Transformer Models

In [ ]:
class BreathBiGRU(nn.Module):
    """Bidirectional GRU operating on breath-level physiological feature tokens."""
    def __init__(self, in_features=NUM_FEATURES, hidden_dim=96, num_layers=2, num_classes=5, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, pad_mask=None):
        h = self.input_proj(x)
        out, _ = self.gru(h)
        return self.head(out)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class BreathTransformer(nn.Module):
    """Transformer Encoder using global self-attention across all breath tokens in a case."""
    def __init__(self, in_features=NUM_FEATURES, d_model=128, nhead=4, num_layers=3, num_classes=5, dropout=0.15):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len=1500)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, x, pad_mask=None):
        h = self.pos_enc(self.input_proj(x))
        out = self.encoder(h, src_key_padding_mask=pad_mask)
        return self.head(out)

## Inference code for test evaluation (128 Hz Unified Resolution)

In [ ]:
@torch.no_grad()
def predict_full_cases(model, X, mean, std, window_samples=7680, stride_samples=1920, batch_size=32):
    """
    Predicts full-length recordings (N_cases, total_len) for 1D U-Net by averaging overlapping
    window softmax probabilities so boundaries are smooth and artifact-free.
    """
    model.eval()
    n_cases, _, total_len = X.shape
    num_classes = model.head.out_channels

    prob_accum = np.zeros((n_cases, num_classes, total_len), dtype=np.float32)
    weight_accum = np.zeros((1, 1, total_len), dtype=np.float32)

    win_weight = np.hanning(window_samples).astype(np.float32) + 0.1
    win_weight = win_weight[None, None, :]

    starts = list(range(0, total_len - window_samples + 1, stride_samples))
    if starts[-1] + window_samples < total_len:
        starts.append(total_len - window_samples)

    mean_b = mean[None, :, None]
    std_b = std[None, :, None]

    for start in starts:
        end = start + window_samples
        batch_np = (X[:, :, start:end] - mean_b) / (std_b + 1e-6)

        probs_list = []
        for b_idx in range(0, n_cases, batch_size):
            xb = torch.from_numpy(batch_np[b_idx:b_idx + batch_size]).float().to(DEVICE)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_list.append(probs)

        all_probs = np.concatenate(probs_list, axis=0)
        prob_accum[:, :, start:end] += all_probs * win_weight
        weight_accum[:, :, start:end] += win_weight

    avg_probs = prob_accum / np.maximum(weight_accum, 1e-6)
    preds = np.argmax(avg_probs, axis=1)
    return preds, avg_probs


@torch.no_grad()
def predict_full_cases_token_model(model, feats_list, segments_list, total_len, feat_mean, feat_std):
    """
    Predicts breath token sequences for BiGRU / Transformer and maps token predictions
    back to the exact 128 Hz sample grid so evaluation matches 1D U-Net sample-for-sample.
    """
    model.eval()
    n_cases = len(feats_list)
    preds_128hz = np.zeros((n_cases, total_len), dtype=np.int64)

    for i in range(n_cases):
        norm_f = (feats_list[i] - feat_mean) / (feat_std + 1e-6)
        xb = torch.from_numpy(norm_f).unsqueeze(0).float().to(DEVICE)
        logits = model(xb)
        token_preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy()
        for tok_idx, (s, e) in enumerate(segments_list[i]):
            preds_128hz[i, s:e] = token_preds[tok_idx]

    return preds_128hz

## Training, Evaluation & Head-to-Head Comparison

In [ ]:
def train_and_evaluate(
    unet_epochs=30,
    token_epochs=35,
    batch_size=32,
    unet_lr=1e-3,
    token_lr=1.5e-3,
    window_sec=60,
    train_stride_sec=30,
    base_filters=32,
):
    # Load metadata & data
    with open(os.path.join(DATASET_DIR, "metadata.json"), "r") as f:
        metadata = json.load(f)

    fs = metadata["fs"]
    channels = metadata["channels"]
    regime_names = {int(k): v for k, v in metadata["regimes"].items()}
    num_classes = len(regime_names)
    target_names = [regime_names[i] for i in range(num_classes)]

    mean = np.array([metadata["train_normalization"]["mean"][c] for c in channels], dtype=np.float32)
    std = np.array([metadata["train_normalization"]["std"][c] for c in channels], dtype=np.float32)

    print("Loading dataset splits...")
    train_npz = np.load(os.path.join(DATASET_DIR, "train.npz"))
    val_npz = np.load(os.path.join(DATASET_DIR, "val.npz"))
    test_npz = np.load(os.path.join(DATASET_DIR, "test.npz"))

    X_train, y_train = train_npz["X"], train_npz["y"]
    X_val, y_val = val_npz["X"], val_npz["y"]
    X_test, y_test = test_npz["X"], test_npz["y"]
    total_len = X_test.shape[-1]

    all_results = {}

    # ==========================================================================
    # 1. TRAIN & EVALUATE 1D U-NET BASELINE
    # ==========================================================================
    print("\n" + "=" * 70 + "\n[1/3] Training 1D U-Net Baseline...\n" + "=" * 70)
    win_samples = int(window_sec * fs)
    stride_samples = int(train_stride_sec * fs)

    train_ds = AnesthesiaWindowDataset(X_train, y_train, mean, std, win_samples, stride_samples)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    _, class_counts = np.unique(y_train, return_counts=True)
    class_weights = (class_counts.sum() / (num_classes * class_counts)) ** 0.5
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
    print(f"Sample class weights: {dict(zip(regime_names.values(), np.round(class_weights, 3)))}")

    model_unet = UNet1D(in_channels=3, num_classes=num_classes, base_filters=base_filters, dropout=0.1).to(DEVICE)
    optimizer = torch.optim.AdamW(model_unet.parameters(), lr=unet_lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=unet_epochs)
    criterion = nn.CrossEntropyLoss(weight=class_weights_t)

    best_val_f1 = -1.0
    history_unet = []
    unet_weights_path = os.path.join(SAVE_DIR, "best_unet1d.pt")

    for epoch in range(1, unet_epochs + 1):
        t0 = time.time()
        model_unet.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model_unet(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        scheduler.step()
        train_loss = running_loss / len(train_ds)

        val_preds, _ = predict_full_cases(model_unet, X_val, mean, std, window_samples=win_samples, stride_samples=win_samples // 2)
        val_macro_f1 = float(f1_score(y_val.ravel(), val_preds.ravel(), average="macro"))
        val_acc = float((val_preds == y_val).mean())

        elapsed = time.time() - t0
        print(
            f"[UNet1D] Epoch {epoch:02d}/{unet_epochs:02d} [{elapsed:.1f}s] | "
            f"Train Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro-F1: {val_macro_f1:.4f}"
        )
        history_unet.append({"epoch": epoch, "train_loss": train_loss, "val_acc": val_acc, "val_macro_f1": val_macro_f1})

        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(
                {
                    "model_name": "UNet1D",
                    "model_state_dict": model_unet.state_dict(),
                    "model_config": {
                        "in_channels": 3,
                        "num_classes": num_classes,
                        "base_filters": base_filters,
                        "window_samples": win_samples,
                    },
                    "normalization": {"mean": mean.tolist(), "std": std.tolist()},
                    "regime_names": regime_names,
                    "fs": fs,
                    "best_val_macro_f1": best_val_f1,
                },
                unet_weights_path,
            )

    ckpt = torch.load(unet_weights_path, map_location=DEVICE)
    model_unet.load_state_dict(ckpt["model_state_dict"])
    t_inf0 = time.time()
    test_preds_unet, _ = predict_full_cases(
        model_unet, X_test, mean, std, window_samples=win_samples, stride_samples=win_samples // 4
    )
    unet_inf_ms = ((time.time() - t_inf0) / len(X_test)) * 1000.0

    report_dict = classification_report(y_test.ravel(), test_preds_unet.ravel(), target_names=target_names, output_dict=True, digits=4)
    report_text = classification_report(y_test.ravel(), test_preds_unet.ravel(), target_names=target_names, digits=4)
    cm_unet = confusion_matrix(y_test.ravel(), test_preds_unet.ravel())

    print("\n--- 1D U-NET TEST CLASSIFICATION REPORT ---")
    print(report_text)

    all_results["UNet1D"] = {
        "best_val_macro_f1": best_val_f1,
        "test_overall_accuracy": float((y_test == test_preds_unet).mean()),
        "test_macro_f1": float(report_dict["macro avg"]["f1-score"]),
        "inference_ms_per_case": round(unet_inf_ms, 2),
        "per_class_metrics": report_dict,
        "confusion_matrix": cm_unet.tolist(),
        "training_history": history_unet,
    }

    # ==========================================================================
    # 2. TOKENIZE DATASET & TRAIN/EVALUATE BreathBiGRU AND BreathTransformer
    # ==========================================================================
    print("\nExtracting breath-level tokens for BiGRU and Transformer...")
    t_tok0 = time.time()
    train_feats, train_y_tok, _ = tokenize_split(X_train, y_train, fs=fs)
    val_feats, _, val_segs = tokenize_split(X_val, y_val, fs=fs)
    test_feats, _, test_segs = tokenize_split(X_test, y_test, fs=fs)
    print(f"Tokenization complete in {time.time() - t_tok0:.1f}s (avg {np.mean([len(f) for f in train_feats]):.1f} tokens/case)")

    all_train_f = np.concatenate(train_feats, axis=0)
    feat_mean = all_train_f.mean(axis=0).astype(np.float32)
    feat_std = all_train_f.std(axis=0).astype(np.float32)

    tok_train_ds = BreathSequenceDataset(train_feats, train_y_tok, feat_mean, feat_std)
    tok_train_loader = DataLoader(tok_train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_breath_sequences)

    all_train_y_tok = np.concatenate(train_y_tok)
    _, tok_counts = np.unique(all_train_y_tok, return_counts=True)
    tok_weights = torch.tensor((tok_counts.sum() / (num_classes * tok_counts)) ** 0.5, dtype=torch.float32, device=DEVICE)
    tok_criterion = nn.CrossEntropyLoss(weight=tok_weights, ignore_index=-100)

    token_models = {
        "BreathBiGRU": (
            BreathBiGRU(in_features=NUM_FEATURES, hidden_dim=96, num_layers=2, num_classes=num_classes),
            {"in_features": NUM_FEATURES, "hidden_dim": 96, "num_layers": 2, "num_classes": num_classes},
        ),
        "BreathTransformer": (
            BreathTransformer(in_features=NUM_FEATURES, d_model=128, nhead=4, num_layers=3, num_classes=num_classes),
            {"in_features": NUM_FEATURES, "d_model": 128, "nhead": 4, "num_layers": 3, "num_classes": num_classes},
        ),
    }

    for idx_m, (model_name, (model, m_cfg)) in enumerate(token_models.items(), start=2):
        print("\n" + "=" * 70 + f"\n[{idx_m}/3] Training {model_name}...\n" + "=" * 70)
        model = model.to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=token_lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=token_epochs)

        best_tok_val_f1 = -1.0
        tok_history = []
        w_path = os.path.join(SAVE_DIR, f"best_{model_name.lower()}.pt")

        for epoch in range(1, token_epochs + 1):
            t0 = time.time()
            model.train()
            run_loss = 0.0
            for xb, yb, pad_mask in tok_train_loader:
                xb, yb, pad_mask = xb.to(DEVICE), yb.to(DEVICE), pad_mask.to(DEVICE)
                opt.zero_grad()
                logits = model(xb, pad_mask=pad_mask)
                loss = tok_criterion(logits.reshape(-1, num_classes), yb.reshape(-1))
                loss.backward()
                opt.step()
                run_loss += loss.item() * xb.size(0)
            sched.step()

            val_preds_128 = predict_full_cases_token_model(model, val_feats, val_segs, total_len, feat_mean, feat_std)
            val_f1 = float(f1_score(y_val.ravel(), val_preds_128.ravel(), average="macro"))
            val_acc = float((val_preds_128 == y_val).mean())
            tok_history.append({"epoch": epoch, "train_loss": run_loss / len(tok_train_ds), "val_acc": val_acc, "val_macro_f1": val_f1})

            if epoch % 5 == 0 or epoch == token_epochs:
                print(
                    f"[{model_name}] Epoch {epoch:02d}/{token_epochs:02d} [{time.time()-t0:.1f}s] | "
                    f"Train Loss: {run_loss/len(tok_train_ds):.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro-F1: {val_f1:.4f}"
                )

            if val_f1 > best_tok_val_f1:
                best_tok_val_f1 = val_f1
                torch.save(
                    {
                        "model_name": model_name,
                        "model_state_dict": model.state_dict(),
                        "model_config": m_cfg,
                        "feat_mean": feat_mean.tolist(),
                        "feat_std": feat_std.tolist(),
                        "feature_names": FEATURE_NAMES,
                        "regime_names": regime_names,
                        "fs": fs,
                        "best_val_macro_f1": best_tok_val_f1,
                    },
                    w_path,
                )

        ckpt = torch.load(w_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        t_inf0 = time.time()
        test_preds_128 = predict_full_cases_token_model(model, test_feats, test_segs, total_len, feat_mean, feat_std)
        inf_ms = ((time.time() - t_inf0) / len(X_test)) * 1000.0

        rep_dict = classification_report(y_test.ravel(), test_preds_128.ravel(), target_names=target_names, output_dict=True, digits=4)
        rep_text = classification_report(y_test.ravel(), test_preds_128.ravel(), target_names=target_names, digits=4)
        cm_tok = confusion_matrix(y_test.ravel(), test_preds_128.ravel())

        print(f"\n--- {model_name} TEST CLASSIFICATION REPORT (128 Hz) ---")
        print(rep_text)

        all_results[model_name] = {
            "best_val_macro_f1": best_tok_val_f1,
            "test_overall_accuracy": float((y_test == test_preds_128).mean()),
            "test_macro_f1": float(rep_dict["macro avg"]["f1-score"]),
            "inference_ms_per_case": round(inf_ms, 2),
            "per_class_metrics": rep_dict,
            "confusion_matrix": cm_tok.tolist(),
            "training_history": tok_history,
        }

    # ==========================================================================
    # 3. HEAD-TO-HEAD MODEL COMPARISON SUMMARY, PLOTS & EXPORT
    # ==========================================================================
    print("\n" + "=" * 88)
    print("FINAL TEST SET BENCHMARK COMPARISON (All Models Evaluated at 128 Hz)")
    print("=" * 88)
    print(f"{'Model':<20} | {'Val Macro-F1':<12} | {'Test Acc':<10} | {'Test Macro-F1':<13} | {'Inf (ms/case)':<13}")
    print("-" * 88)
    for m_name, res in all_results.items():
        print(
            f"{m_name:<20} | {res['best_val_macro_f1']:<12.4f} | "
            f"{res['test_overall_accuracy']*100:<9.2f}% | {res['test_macro_f1']:<13.4f} | {res['inference_ms_per_case']:<13.2f}"
        )

    print("\nPer-Class Test F1 Scores:")
    cls_header = f"{'Model':<20} | " + " | ".join([f"{n[:13]:<13}" for n in target_names])
    print(cls_header)
    print("-" * len(cls_header))
    for m_name, res in all_results.items():
        row_vals = " | ".join([f"{res['per_class_metrics'][n]['f1-score']:<13.4f}" for n in target_names])
        print(f"{m_name:<20} | {row_vals}")

    # Save JSON evaluation results (no 'artifacts' suffix)
    eval_json_path = os.path.join(SAVE_DIR, "eval_results.json")
    with open(eval_json_path, "w") as f:
        json.dump(all_results, f, indent=2)

    # Plot side-by-side normalized confusion matrices + per-class F1 bar comparison
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.15, 1.0], hspace=0.38, wspace=0.25)

    for idx, (m_name, res) in enumerate(all_results.items()):
        ax = fig.add_subplot(gs[0, idx])
        cm = np.array(res["confusion_matrix"], dtype=float)
        cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1.0)
        im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(num_classes))
        ax.set_yticks(range(num_classes))
        ax.set_xticklabels([n[:10] for n in target_names], rotation=30, ha="right", fontsize=8.5)
        ax.set_yticklabels([n[:10] for n in target_names], fontsize=8.5)
        ax.set_title(f"{m_name}\nTest Macro-F1: {res['test_macro_f1']:.4f}", fontsize=10.5)
        if idx == 0:
            ax.set_ylabel("True Regime")
        ax.set_xlabel("Predicted Regime")

        for i in range(num_classes):
            for j in range(num_classes):
                ax.text(j, i, f"{cm_norm[i, j]*100:.1f}%", ha="center", va="center",
                        color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=8)

    ax_bar = fig.add_subplot(gs[1, :])
    x_pos = np.arange(num_classes)
    width = 0.25
    bar_colors = ["#3498db", "#e67e22", "#2ecc71"]
    for idx, (m_name, res) in enumerate(all_results.items()):
        scores = [res["per_class_metrics"][n]["f1-score"] for n in target_names]
        bars = ax_bar.bar(x_pos + (idx - 1) * width, scores, width, label=m_name, color=bar_colors[idx])
        for bar in bars:
            h = bar.get_height()
            ax_bar.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}", ha="center", va="bottom", fontsize=8.5)

    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(target_names, fontsize=10)
    ax_bar.set_ylim(0, 1.12)
    ax_bar.set_ylabel("Test F1 Score")
    ax_bar.set_title("Per-Class Test F1 Score Comparison Across Architectures", fontsize=11.5)
    ax_bar.grid(axis="y", linestyle="--", alpha=0.4)
    ax_bar.legend(loc="lower right", fontsize=9.5)

    comp_png_path = os.path.join(SAVE_DIR, "model_comparison.png")
    plt.savefig(comp_png_path, dpi=150, bbox_inches="tight")
    plt.show()

    # Zip weights & results cleanly without 'artifacts' suffix
    zip_path = os.path.join(BASE_DIR, "ventilation_models.zip")
    os.system(f"zip -r -j -q {zip_path} {SAVE_DIR}")
    print(f"\nAll weights & results saved to {SAVE_DIR} and zipped to: {zip_path}")
    print("In Colab, run: from google.colab import files; files.download('/content/ventilation_models.zip')")
    return all_results

## Run

In [ ]:
results = train_and_evaluate(
    unet_epochs=30,
    token_epochs=35,
    batch_size=32,
    unet_lr=1e-3,
    token_lr=1.5e-3,
)